# 🍌 Banana Analysis Pipeline
### Integrated: Segmentation (Mask R-CNN) → Ripeness Classification (MobileNetV2)

**Workflow:**  
1. Load both trained models  
2. Run Mask R-CNN to detect & segment individual bananas  
3. Crop each segmented banana  
4. Run MobileNetV2 to classify ripeness of each crop  
5. Visualize results with overlays and confidence bars

## 1. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pycocotools --quiet

## 2. Import Libraries

In [ ]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision import models, transforms
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 3. Configuration — Set Your Model Paths Here

In [ ]:
# ── Edit these paths to point to your saved checkpoints ──────────────────
SEG_MODEL_PATH  = "/content/drive/MyDrive/banana_maskrcnn_finetuned.pth"
CLS_MODEL_PATH  = "/content/drive/MyDrive/banana_mobilenetv2_best.pth"

# Detection thresholds
SEG_SCORE_THRESHOLD = 0.90    # Mask R-CNN confidence threshold
CLS_CONFIDENCE_THRESHOLD = 0.60  # MobileNetV2 low-confidence warning

# Padding (pixels) added around each banana crop before classification
CROP_PADDING = 10

print("Config loaded.")
print(f"  Segmentation model : {SEG_MODEL_PATH}")
print(f"  Classification model: {CLS_MODEL_PATH}")

## 4. Load Segmentation Model (Mask R-CNN)

In [ ]:
def load_segmentation_model(path, device):
    """Load fine-tuned Mask R-CNN (2 classes: background + banana)."""
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)

    num_classes = 2  # background + banana

    # Replace box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = (
        torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            in_features, num_classes
        )
    )

    # Replace mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = (
        torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
            in_features_mask, 256, num_classes
        )
    )

    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Segmentation model loaded from: {path}")
    return model

seg_model = load_segmentation_model(SEG_MODEL_PATH, device)

## 5. Load Classification Model (MobileNetV2)

In [ ]:
def load_classification_model(path, device, num_classes=4):
    """Load fine-tuned MobileNetV2 ripeness classifier."""
    model = models.mobilenet_v2(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Classification model loaded from: {path}")
    return model

# Class mapping  {idx: label}  — must match training order
IDX_TO_CLASS = {0: "overripe", 1: "ripe", 2: "rotten", 3: "unripe"}

cls_model = load_classification_model(CLS_MODEL_PATH, device)

## 6. Transforms & Helper Functions

In [ ]:
# ── Inference transform for the classifier (no augmentation) ─────────────
inference_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── TTA transforms (5 variants, same as training notebook) ───────────────
tta_transforms = [
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize(280), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.RandomRotation(degrees=(10, 10)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.RandomRotation(degrees=(-10, -10)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
]


def auto_correct_exposure(pil_img):
    """Normalize extreme brightness before classification inference."""
    arr  = np.array(pil_img.convert('L'))
    mean = arr.mean()
    tag  = ""
    if mean < 60:
        factor  = min(2.5, 120.0 / (mean + 1e-6))
        pil_img = ImageEnhance.Brightness(pil_img).enhance(factor)
        tag = f" [low-light corrected, orig_mean={mean:.0f}]"
    elif mean > 200:
        factor  = max(0.5, 160.0 / (mean + 1e-6))
        pil_img = ImageEnhance.Brightness(pil_img).enhance(factor)
        tag = f" [high-light corrected, orig_mean={mean:.0f}]"
    return pil_img, tag


def predict_ripeness_with_tta(pil_img, model, device, tta_transforms):
    """Average softmax probabilities across TTA variants."""
    all_probs = []
    with torch.no_grad():
        for tfm in tta_transforms:
            tensor = tfm(pil_img).unsqueeze(0).to(device)
            logits = model(tensor)
            probs  = F.softmax(logits, dim=1)
            all_probs.append(probs)
    avg_probs = torch.stack(all_probs).mean(dim=0)
    confidence, pred_idx = torch.max(avg_probs, dim=1)
    return pred_idx.item(), confidence.item(), avg_probs.squeeze().cpu().numpy()

## 7. Integrated Pipeline Function

In [ ]:
def run_banana_pipeline(image_path,
                        seg_model,
                        cls_model,
                        device,
                        seg_threshold=SEG_SCORE_THRESHOLD,
                        crop_padding=CROP_PADDING):
    """
    Full pipeline:
      1. Read image
      2. Run Mask R-CNN segmentation
      3. For each detected banana, crop the bounding box region
      4. Classify ripeness with MobileNetV2 + TTA
      5. Return annotated image and per-banana results list

    Returns
    -------
    image_rgb : np.ndarray  (H, W, 3) original image
    results   : list of dicts with keys:
                  banana_id, score, mask, box, crop_pil,
                  ripeness, confidence, all_probs, exposure_tag
    """
    # ── Load image ────────────────────────────────────────────────────────
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]

    # ── Segmentation ──────────────────────────────────────────────────────
    img_tensor = T.ToTensor()(image_rgb).to(device)
    with torch.no_grad():
        prediction = seg_model([img_tensor])

    labels  = prediction[0]['labels']
    scores  = prediction[0]['scores']
    masks   = prediction[0]['masks']
    boxes   = prediction[0]['boxes']

    BANANA_CLASS_ID = 1
    keep = (labels == BANANA_CLASS_ID) & (scores > seg_threshold)

    banana_masks  = masks[keep]
    banana_scores = scores[keep]
    banana_boxes  = boxes[keep]

    print(f"Bananas detected: {len(banana_masks)}")

    results = []

    for i, (mask, score, box) in enumerate(
            zip(banana_masks, banana_scores, banana_boxes)):

        binary_mask = (mask[0] > 0.5).cpu().numpy()  # (H, W) bool

        # ── Crop bounding box (with padding) ──────────────────────────────
        x1, y1, x2, y2 = box.cpu().numpy().astype(int)
        x1 = max(0, x1 - crop_padding)
        y1 = max(0, y1 - crop_padding)
        x2 = min(w, x2 + crop_padding)
        y2 = min(h, y2 + crop_padding)

        crop_np  = image_rgb[y1:y2, x1:x2]
        crop_pil = Image.fromarray(crop_np)

        # ── Exposure correction + classify ────────────────────────────────
        corrected_crop, exposure_tag = auto_correct_exposure(crop_pil)
        pred_idx, confidence, all_probs = predict_ripeness_with_tta(
            corrected_crop, cls_model, device, tta_transforms
        )

        ripeness = IDX_TO_CLASS[pred_idx]

        results.append({
            "banana_id"    : i + 1,
            "seg_score"    : score.item(),
            "mask"         : binary_mask,
            "box"          : (x1, y1, x2, y2),
            "crop_pil"     : crop_pil,
            "ripeness"     : ripeness,
            "confidence"   : confidence,
            "all_probs"    : all_probs,
            "exposure_tag" : exposure_tag,
        })

        warning = " ⚠️ LOW CONF" if confidence < CLS_CONFIDENCE_THRESHOLD else ""
        print(f"  Banana {i+1}: {ripeness}  "
              f"(cls={confidence*100:.1f}%{warning}, "
              f"seg={score.item():.2f}){exposure_tag}")

    return image_rgb, results

## 8. Visualisation Functions

In [ ]:
# ── Colour map for ripeness labels ───────────────────────────────────────
RIPENESS_COLORS = {
    "unripe"   : (0,   200,  50),   # green
    "ripe"     : (255, 220,   0),   # yellow
    "overripe" : (255, 140,   0),   # orange
    "rotten"   : (180,   0,   0),   # dark red
}


def visualise_results(image_rgb, results):
    """
    Two-panel figure per detected banana:
      Left  : cropped banana image
      Right : per-class probability bar chart

    Plus one full-image overlay showing coloured masks + labels.
    """
    if not results:
        print("No bananas detected — nothing to visualise.")
        return

    # ── Full-image overlay ────────────────────────────────────────────────
    overlay = image_rgb.copy()
    legend_patches = []

    for res in results:
        color = RIPENESS_COLORS.get(res["ripeness"], (128, 128, 128))
        mask  = res["mask"]
        x1, y1, x2, y2 = res["box"]

        # Semi-transparent coloured mask
        overlay[mask] = (
            overlay[mask] * 0.45 + np.array(color) * 0.55
        ).astype(np.uint8)

        # Bounding box
        cv2.rectangle(overlay, (x1, y1), (x2, y2),
                      color[::-1], 2)   # OpenCV uses BGR... overlay is RGB so keep RGB

        # Label text
        label_txt = (f"#{res['banana_id']} {res['ripeness']} "
                     f"{res['confidence']*100:.0f}%")
        cv2.putText(overlay, label_txt, (x1, max(y1 - 8, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    color, 2, cv2.LINE_AA)

        legend_patches.append(
            mpatches.Patch(color=[c/255 for c in color],
                           label=f"#{res['banana_id']} — {res['ripeness']}")
        )

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(overlay)
    ax.set_title("Banana Segmentation + Ripeness Classification", fontsize=14)
    ax.axis("off")
    ax.legend(handles=legend_patches, loc="upper right",
              fontsize=10, framealpha=0.8)
    plt.tight_layout()
    plt.show()

    # ── Per-banana detail panels ──────────────────────────────────────────
    class_names = [IDX_TO_CLASS[i] for i in range(4)]

    for res in results:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        # Left: crop
        axes[0].imshow(res["crop_pil"])
        warning = " ⚠️ LOW CONFIDENCE" if res["confidence"] < CLS_CONFIDENCE_THRESHOLD else ""
        title = (f"Banana #{res['banana_id']} — {res['ripeness']} "
                 f"({res['confidence']*100:.1f}%){warning}{res['exposure_tag']}")
        axes[0].set_title(title, fontsize=10, fontweight="bold", wrap=True)
        axes[0].axis("off")

        # Right: probability bar chart
        color_list = [
            [c/255 for c in RIPENESS_COLORS.get(cls, (128,128,128))]
            for cls in class_names
        ]
        axes[1].barh(class_names, res["all_probs"] * 100, color=color_list)
        axes[1].set_xlabel("Probability (%)")
        axes[1].set_title("Ripeness Class Probabilities (TTA)")
        axes[1].set_xlim(0, 100)
        for j, v in enumerate(res["all_probs"] * 100):
            axes[1].text(v + 0.5, j, f"{v:.1f}%", va="center", fontsize=10)

        plt.tight_layout()
        plt.show()


def print_summary(results):
    """Print a compact text summary of all detected bananas."""
    print("\n" + "="*55)
    print(f"{'BANANA ANALYSIS SUMMARY':^55}")
    print("="*55)
    print(f"{'#':<6}{'Ripeness':<12}{'Cls Conf':>10}{'Seg Score':>12}")
    print("-"*55)
    for r in results:
        warning = " ⚠️" if r["confidence"] < CLS_CONFIDENCE_THRESHOLD else ""
        print(f"{r['banana_id']:<6}{r['ripeness']:<12}"
              f"{r['confidence']*100:>9.1f}%"
              f"{r['seg_score']:>11.2f}{warning}")
    print("="*55)

    # Ripeness distribution
    from collections import Counter
    counts = Counter(r["ripeness"] for r in results)
    print("\nRipeness Distribution:")
    for label, n in sorted(counts.items()):
        print(f"  {label:<10}: {n}")
    print()

## 9. Run the Pipeline

### Option A — Test on a single image file

In [ ]:
# ── Provide the path to your test image ──────────────────────────────────
TEST_IMAGE_PATH = "/content/drive/MyDrive/test.jpg"   # ← change as needed

image_rgb, results = run_banana_pipeline(
    TEST_IMAGE_PATH,
    seg_model,
    cls_model,
    device
)

visualise_results(image_rgb, results)
print_summary(results)

### Option B — Upload an image interactively (Colab)

In [ ]:
from google.colab import files
import io

print("Upload one or more banana images:")
uploaded = files.upload()

for fname, data in uploaded.items():
    # Save temporarily
    tmp_path = f"/content/{fname}"
    with open(tmp_path, "wb") as f:
        f.write(data)

    print(f"\n{'─'*50}")
    print(f"Processing: {fname}")
    print(f"{'─'*50}")

    image_rgb, results = run_banana_pipeline(
        tmp_path, seg_model, cls_model, device
    )
    visualise_results(image_rgb, results)
    print_summary(results)

### Option C — Batch process a folder of images

In [ ]:
BATCH_FOLDER = "/content/drive/MyDrive/banana_test_images"   # ← change

valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_files = [
    os.path.join(BATCH_FOLDER, f)
    for f in sorted(os.listdir(BATCH_FOLDER))
    if os.path.splitext(f)[1].lower() in valid_ext
]

print(f"Found {len(image_files)} images in {BATCH_FOLDER}\n")

all_results = {}

for img_path in image_files:
    fname = os.path.basename(img_path)
    print(f"\n{'='*55}")
    print(f"Image: {fname}")
    print('='*55)
    try:
        image_rgb, results = run_banana_pipeline(
            img_path, seg_model, cls_model, device
        )
        visualise_results(image_rgb, results)
        print_summary(results)
        all_results[fname] = results
    except Exception as e:
        print(f"  ERROR: {e}")

print("\nBatch processing complete.")